<a href="https://colab.research.google.com/github/sandy-gags/CRE-LLM/blob/main/Copy_of_2026_03_12_queensland_ai_talk_small_language_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Queensland AI Meetup — Fine-Tune Gemma 3 270M Live Demo

End-to-end notebook: load data → fine-tune → evaluate → upload → demo

Run in Google Colab with GPU enabled: Runtime → Change runtime type → G4 GPU

Install deps (run this in a separate cell first):

```
!pip install -q torch transformers datasets trl accelerate huggingface_hub gradio rouge-score matplotlib
```

## What are we going to do?

We're going to play who's here?

A variant of Guess Who.

We'll take a base small language model (Gemma 3 270M) and fine-tune to tell us information about the guests at tonight's event.

Example input:

```
Daniel Bourke
```

Example output:

```
Daniel Bourke is a Brisbane-based Machine Learning Engineer and Instructor at Zero To Mastery, widely known for his self-taught path into the Al industry. He previously applied Al to healthcare and insurance challenges during his time at Max Kelsen. Today, he teaches thousands of students globally through his courses and popular educational platforms like learnpytorch.io. Beyond teaching, Daniel is the co-founder of the food-recognition app Nutrify and has authored a philosophical novel called 'Charlie Walks'. His YouTube channel, which documents his learning journey, has amassed over 5 million views.
Interestingly, he transitioned into tech after working as an Apple Genius and an Uber driver.
```

## Recipe

* Load dataset
* Inspect dataset
* Fine-tune model
* Evaluate model
* Upload model to Hugging Face Hub for reusability
* Create a demo application showing our model working

## Quick links

* Base model - https://huggingface.co/google/gemma-3-270m-it
* Dataset - https://huggingface.co/datasets/mrdbourke/queensland-ai-meetup-sft-v2
* Base model demo - https://huggingface.co/spaces/mrdbourke/queensland-ai-base-gemma3
* Fine-tuned model vs base model demo - https://huggingface.co/spaces/mrdbourke/queensland-ai-fine-tuned-gemma3

In [ ]:
# # If you need to login to Hugging Face
# from huggingface_hub import notebook_login
# notebook_login()

## 0. Install Dependencies

In [ ]:
!pip install -U transformers datasets trl accelerate huggingface_hub gradio rouge-score matplotlib

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 kB 50.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 528.8/528.8 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 118.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 22.7 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=98162afda1f32fb5c3e5217eefc8c96b7c7791a746175674a491a120fb81773e
  Stor

Check for our GPU.

In [ ]:
!nvidia-smi

Tue Mar 17 01:24:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Setup quick links.

In [ ]:
quick_links = {
    "meetup_event_link": "https://www.meetup.com/en-au/queensland_ai/events/313408661/",
    "base_model": "https://huggingface.co/google/gemma-3-270m-it",
    "dataset": "https://huggingface.co/datasets/mrdbourke/queensland-ai-meetup-sft-v2",
    "base_model_demo": "https://huggingface.co/spaces/mrdbourke/queensland-ai-base-gemma3",
    "fine_tuned_model_demo": "https://huggingface.co/spaces/mrdbourke/queensland-ai-fine-tuned-gemma3",
    "name":"gauranga magriplis" # turns out this person was on the Meetup attendee list 😂
}


## 1. Imports + another hardware check (for fun)


In [ ]:
import json
import os
import random
import time

import matplotlib.pyplot as plt
import torch
import transformers
import trl

print(f"[INFO] PyTorch version: {torch.__version__}")
print(f"[INFO] Transformers version: {transformers.__version__}")
print(f"[INFO] TRL version: {trl.__version__}")

# Check GPU
if torch.cuda.is_available():
    device = torch.cuda.current_device()
    gpu_name = torch.cuda.get_device_name(device)
    total_mem = torch.cuda.get_device_properties(device).total_memory / 1e9
    print(f"[INFO] GPU: {gpu_name} ({total_mem:.1f} GB)")
else:
    print("[WARN] No CUDA GPU detected! Training will be very slow.")
    print("[WARN] Go to Runtime → Change runtime type → T4 GPU")

[INFO] PyTorch version: 2.10.0+cu128
[INFO] Transformers version: 5.3.0
[INFO] TRL version: 0.29.0
[INFO] GPU: Tesla T4 (15.6 GB)


## 2. Setup base model

* Base model: https://huggingface.co/google/gemma-3-270m-it

> **Note:** Should you fine-tune base (`google/gemma-3-270m`) model or instruction tuned (`-it`) model? Try both. In my practice, I've found not much difference between each.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

MODEL_NAME = "google/gemma-3-270m-it"  # "it" = instruction-tuned

print(f"[INFO] Loading base model: {MODEL_NAME}")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
    attn_implementation="eager",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"[INFO] Model loaded — dtype: {model.dtype}, device: {model.device}")
print(f"[INFO] Parameters: {sum(p.numel() for p in model.parameters()):,}")

[INFO] Loading base model: google/gemma-3-270m-it


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/google/gemma-3-270m-it.
401 Client Error. (Request ID: Root=1-69b8ad70-55f41fa40310613c35fb3e99;4f1a4bda-9056-4030-b77d-40757db7e71a)

Cannot access gated repo for url https://huggingface.co/google/gemma-3-270m-it/resolve/main/config.json.
Access to model google/gemma-3-270m-it is restricted. You must have access to it and be authenticated to access it. Please log in.

## 3. Try the base model (before fine-tuning)

In [ ]:
from transformers import GenerationConfig

base_pipeline = pipeline("text-generation", model=model, tokenizer=tokenizer)

# Helper function
def pred_on_text(pipe, input_text, max_new_tokens=512):
    """Run inference and return (output_text, elapsed_seconds)."""
    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    start = time.time()
    raw = pipe(
        text_inputs=[{"role": "user", "content": input_text}],
        generation_config=gen_config,
    )
    elapsed = round(time.time() - start, 4)
    generated = raw[0]["generated_text"]
    output = generated[-1]["content"] if isinstance(generated, list) else str(generated)
    return output, elapsed

# Test with attendee names + some non-attendee names
test_inputs_before = [
    "Daniel Bourke",
    "William Bourke",
    "What are Daniel Bourke's skills?",
    "Elon Musk",
    "haiku",
]

print("=" * 70)
print("  BASE MODEL (before fine-tuning)")
print("=" * 70)
for text in test_inputs_before:
    output, elapsed = pred_on_text(base_pipeline, text)
    print(f"\n  Input:  \"{text}\"")
    print(f"  Output: {output[:200]}{'...' if len(output) > 200 else ''}")
    print(f"  ({elapsed}s)")


## 4. Load the dataset

Dataset = what we do want our model to do? We can control it with data.

Demo:

* Take "Daniel Bourke" info and get gpt-oss-120b to create synthetic data.

Our custom dataset:

* Meetup attendee list names
* Search LinkedIn for names
* Create synthetic dataset of QA pairs based on public LinkedIn details

Your dataset:

* **Small models ideal for:** Business process that happens repeatedly (plenty of samples which often take tedious human hours to complete).
* **Large models ideal for:** Wide ranging general task that needs plenty of nuance (these use cases are as broad as software and are still be found).

**Notes:**

* Plenty of edge cases (we handle these through experimentation)
* Some names don't appear (naturally)
  * How do we handle these?
* What about names who don't appear in the Meetup attendee list (e.g. Elon Musk), what do we do with these?
  * Things you don't want in your model's behaviour are often called **hard negatives**.
    * In Nutrify's case, hard negatives would be images of things which *aren't* food.
* Why v2? Because v1 was too small (~1000 samples) so I scaled things up to ~8k.

In [ ]:
from datasets import load_dataset

DATASET_NAME = "mrdbourke/queensland-ai-meetup-sft-v2"

dataset = load_dataset(DATASET_NAME)

print(f"[INFO] Dataset: {DATASET_NAME}")
print(f"[INFO] Train samples: {len(dataset['train'])}")
print(f"[INFO] Test samples:  {len(dataset['test'])}")


## 5. Inspect dataset samples


In [ ]:
# Show a few examples of what the model will learn
print("\n" + "=" * 70)
print("  TRAINING EXAMPLES")
print("=" * 70)

# Pick a few representative examples
for i in [0, len(dataset["train"]) // 4, len(dataset["train"]) // 2]:
    sample = dataset["train"][i]
    user_msg = sample["messages"][0]["content"]
    model_msg = sample["messages"][1]["content"]
    print(f"\n  [{i}] User:  \"{user_msg[:80]}{'...' if len(user_msg) > 80 else ''}\"")
    print(f"       Model: \"{model_msg[:120]}{'...' if len(model_msg) > 120 else ''}\"")

# Show a random example in full
random_idx = random.randint(0, len(dataset["train"]) - 1)
random_sample = dataset["train"][random_idx]
print(f"\n  Full random example (index {random_idx}):")
print(json.dumps(random_sample["messages"], indent=2, ensure_ascii=False))

## 6. Fine-tune with SFTTrainer

SFT = Supervised Fine-tuning, examples of inputs and outputs.


In [ ]:
from trl import SFTConfig, SFTTrainer

CHECKPOINT_DIR_NAME = "./checkpoint_models"
BASE_LEARNING_RATE = 5e-5
NUM_EPOCHS = 2  # 2 epochs is our sweet spot — trains fast, generalises well
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
MAX_LENGTH = 512

torch_dtype = model.dtype

print(f"[INFO] Using dtype: {torch_dtype}")
print(f"[INFO] Epochs: {NUM_EPOCHS}")
print(f"[INFO] Learning rate: {BASE_LEARNING_RATE}")
print(f"[INFO] Batch size: {TRAIN_BATCH_SIZE}")
print(f"[INFO] Max length: {MAX_LENGTH}")

sft_config = SFTConfig(
    output_dir=CHECKPOINT_DIR_NAME,
    max_length=MAX_LENGTH,
    packing=False,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_checkpointing=False,
    optim="adamw_torch_fused",
    logging_steps=1,
    save_strategy="epoch",
    eval_strategy="epoch",
    metric_for_best_model="eval_loss",
    learning_rate=BASE_LEARNING_RATE,
    fp16=(torch_dtype == torch.float16),
    bf16=(torch_dtype == torch.bfloat16),
    lr_scheduler_type="constant",
    push_to_hub=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    processing_class=tokenizer,
)

# Train!
print(f"\n[INFO] Starting fine-tuning...")
training_start = time.time()
training_output = trainer.train()
training_time = round(time.time() - training_start, 1)

print(f"\n[INFO] Fine-tuning complete in {training_time}s!")
print(f"[INFO] Train loss: {training_output.metrics.get('train_loss', 'N/A'):.4f}")

## 7. Evaluate our model's loss curves

In [ ]:
log_history = trainer.state.log_history

train_losses = [log["loss"] for log in log_history if "loss" in log]
epoch_train = [log["epoch"] for log in log_history if "loss" in log]
eval_losses = [log["eval_loss"] for log in log_history if "eval_loss" in log]
epoch_eval = [log["epoch"] for log in log_history if "eval_loss" in log]

plt.figure(figsize=(10, 5))
plt.plot(epoch_train, train_losses, label="Training Loss")
plt.plot(epoch_eval, eval_losses, label="Validation Loss", marker="o")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Fine-Tuning Loss Curves — Gemma 3 270M on Queensland AI Meetup Data")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("loss_curves.png", dpi=150)
plt.show()

# Print final eval loss from training log (no separate evaluate() call needed)
if eval_losses:
    print(f"[INFO] Final eval loss: {eval_losses[-1]:.4f}")
print(f"[INFO] Final train loss: {train_losses[-1]:.4f}")

## 8. Save the best model

In [ ]:
trainer.save_model()
print(f"[INFO] Model saved to: {CHECKPOINT_DIR_NAME}")

# Optional: clean up checkpoint folders (keep only the final model files)
import shutil
for item in os.listdir(CHECKPOINT_DIR_NAME):
    if item.startswith("checkpoint-"):
        shutil.rmtree(os.path.join(CHECKPOINT_DIR_NAME, item))
        print(f"[INFO] Removed {item}")

print(f"[INFO] Files in {CHECKPOINT_DIR_NAME}:")
for f in sorted(os.listdir(CHECKPOINT_DIR_NAME)):
    size = os.path.getsize(os.path.join(CHECKPOINT_DIR_NAME, f)) / 1e6
    print(f"  {f:40s} {size:>8.1f} MB")


## 9. Reload both our models (base and fine-tuned) for comparison

In [ ]:
# SFTTrainer modifies `model` in-place, so `base_pipeline` now points at
# fine-tuned weights. We must reload both from scratch for a fair comparison.

import gc

# Free the training objects first
del model, trainer, base_pipeline
gc.collect()
torch.cuda.empty_cache()
print(f"[INFO] Cleared training objects from GPU memory")

# Load base model fresh (from HF Hub — original weights)
print(f"[INFO] Loading base model fresh: {MODEL_NAME}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype="auto", device_map="auto", attn_implementation="eager")
base_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_pipe = pipeline("text-generation", model=base_model, tokenizer=base_tokenizer)

# Load fine-tuned model fresh (from checkpoint — trained weights)
print(f"[INFO] Loading fine-tuned model: {CHECKPOINT_DIR_NAME}")
ft_model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT_DIR_NAME, dtype="auto", device_map="auto", attn_implementation="eager")
ft_tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT_DIR_NAME)
ft_pipe = pipeline("text-generation", model=ft_model, tokenizer=ft_tokenizer)

print(f"[INFO] Both models loaded — ready for side-by-side comparison")

## 10. Before and after fine-tuning comparison


In [ ]:
from IPython.display import display, HTML
import html as html_mod

comparison_inputs = [
    "Daniel Bourke",
    "daniel bourke",
    "William Bourke",
    "What should I ask Daniel Bourke about?",
    "What should I ask William Bourke about?",
    "Elon Musk",
    "elon musk",
    "Shrek",
    "Michael Tremeer",
    "Leah H.",
    "Adam Luff",
    "Who's here?",
]

# Display header immediately
display(HTML("""
<div style="max-width:1000px; margin:0 auto;">
  <h2 style="text-align:center; color:#fff;">🧠 Before vs After Fine-Tuning</h2>
  <p style="text-align:center; color:#fff;">Same 270M parameter model — left is base weights, right is after 2 epochs on ~7,700 training pairs</p>
</div>
"""))

# Display each card as it completes
comparison_results = []
for i, text in enumerate(comparison_inputs):
    base_out, base_time = pred_on_text(base_pipe, text)
    ft_out, ft_time = pred_on_text(ft_pipe, text)
    comparison_results.append({
        "input": text,
        "base_out": base_out,
        "base_time": base_time,
        "ft_out": ft_out,
        "ft_time": ft_time,
    })
    print(f"  [{i + 1}/{len(comparison_inputs)}] \"{text}\" done")

    # Render this card immediately
    inp = html_mod.escape(text)
    base = html_mod.escape(base_out).replace("\n", "<br>")
    ft = html_mod.escape(ft_out).replace("\n", "<br>")

    display(HTML(f"""
    <div style="max-width:1000px; margin:0 auto;">
      <div style="border:1px solid #ddd; border-radius:12px; padding:20px; margin:16px 0; background:#fafafa;">
        <div style="font-size:14px; color:#000; margin-bottom:4px;">Input</div>
        <div style="font-size:18px; font-weight:600; color:#000; padding:10px 14px; background:#fff; border:2px solid #f60; border-radius:8px; margin-bottom:16px;">{inp}</div>
        <div style="display:flex; gap:16px;">
          <div style="flex:1;">
            <div style="font-size:15px; font-weight:600; color:#000; margin-bottom:8px;">❌ Base Model <span style="font-weight:400; color:#000;">(google/gemma-3-270m-it)</span></div>
            <div style="background:#fff; border:1px solid #ddd; border-radius:8px; padding:12px; font-size:14px; line-height:1.5; min-height:80px; color:#000;">{base}</div>
            <div style="color:#000; font-size:12px; margin-top:6px;">{base_time}s</div>
          </div>
          <div style="flex:1;">
            <div style="font-size:15px; font-weight:600; color:#000; margin-bottom:8px;">✅ Fine-Tuned Model</div>
            <div style="background:#fff; border:1px solid #ddd; border-radius:8px; padding:12px; font-size:14px; line-height:1.5; min-height:80px; color:#000;">{ft}</div>
            <div style="color:#000; font-size:12px; margin-top:6px;">{ft_time}s</div>
          </div>
        </div>
      </div>
    </div>
    """))

How would we improve the inputs/outputs?

What about the "Who's here?" case?

What happened there?

## 11. Evaluate on the test set (ROGUE-L score)

ROGUE-L score = a similarity measure for longest common sequence between two sequences, higher is better.

In [ ]:
from tqdm.auto import tqdm
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

results = []
test_data = list(dataset["test"])

print(f"\n[INFO] Evaluating on {len(test_data)} test samples...")

for i, row in tqdm(enumerate(test_data), desc="Evaluating test samples", total=len(test_data)):
    input_text = row["messages"][0]["content"]
    expected = row["messages"][1]["content"]
    predicted, elapsed = pred_on_text(ft_pipe, input_text)

    rouge = scorer.score(expected, predicted)["rougeL"].fmeasure
    results.append({"input": input_text, "rouge_l": rouge, "elapsed": elapsed})

    if (i + 1) % 25 == 0:
        print(f"  [{i+1}/{len(test_data)}] processed...")

# Summary
rouge_scores = [r["rouge_l"] for r in results]
exact = sum(1 for s in rouge_scores if s >= 0.99)
passed = sum(1 for s in rouge_scores if 0.5 <= s < 0.99)
partial = sum(1 for s in rouge_scores if 0.2 <= s < 0.5)
failed = sum(1 for s in rouge_scores if s < 0.2)

total = len(results)
print(f"\n{'='*60}")
print(f"  Evaluation Summary")
print(f"{'='*60}")
print(f"  Total samples  : {total}")
print(f"  Exact match    : {exact} ({exact/total:.1%})")
print(f"  Pass (≥0.5)    : {passed} ({passed/total:.1%})")
print(f"  Partial (≥0.2) : {partial} ({partial/total:.1%})")
print(f"  Fail (<0.2)    : {failed} ({failed/total:.1%})")
print(f"  Pass+Exact rate: {(exact+passed)/total:.1%}")
print(f"  ROUGE-L mean   : {sum(rouge_scores)/total:.3f}")
print(f"  ROUGE-L median : {sorted(rouge_scores)[total//2]:.3f}")

## 12. Upload the fine-tuned model to Hugging Face Hub

Let's make our artifacts reusable so we can access them whenever we need.

In [ ]:
from huggingface_hub import HfApi, create_repo

SUFFIX = "live"

# NOTE: Update this to your HF username / desired model name
MODEL_REPO_ID = f"mrdbourke/queensland-ai-gemma3-fine-tuned-{SUFFIX}"

api = HfApi()

print(f"[INFO] Creating model repo: {MODEL_REPO_ID}")
create_repo(
    repo_id=MODEL_REPO_ID,
    repo_type="model",
    exist_ok=True,
)

print(f"[INFO] Uploading checkpoint to {MODEL_REPO_ID}...")
api.upload_folder(
    folder_path=CHECKPOINT_DIR_NAME,
    repo_id=MODEL_REPO_ID,
    repo_type="model",
)

print(f"[INFO] ✓ Model uploaded to: https://huggingface.co/{MODEL_REPO_ID}")

## 13. Create Gradio demo comparison app (base model vs fine-tuned)

One of the best things you can do is create a comparison tool (e.g. simple web app or Gradio) to showcase actual examples side by side with different variations/models.

In [ ]:
os.makedirs("demos/comparison", exist_ok=True)

# Write app.py
app_py = '''"""
Queensland AI Meetup — Base vs Fine-Tuned Gemma 3 270M
Side-by-side comparison: same input goes to both models.
"""

import time
import gradio as gr
import spaces
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, pipeline

BASE_MODEL_NAME = "google/gemma-3-270m-it"
FINE_TUNED_MODEL_NAME = "''' + MODEL_REPO_ID + '''"

print("[INFO] Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME, dtype="auto", device_map="auto", attn_implementation="eager")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_pipe = pipeline("text-generation", model=base_model, tokenizer=base_tokenizer)

print("[INFO] Loading fine-tuned model...")
ft_model = AutoModelForCausalLM.from_pretrained(
    FINE_TUNED_MODEL_NAME, dtype="auto", device_map="auto", attn_implementation="eager")
ft_tokenizer = AutoTokenizer.from_pretrained(FINE_TUNED_MODEL_NAME)
ft_pipe = pipeline("text-generation", model=ft_model, tokenizer=ft_tokenizer)
print("[INFO] Both models loaded!")

def run_model(pipe, input_text, max_new_tokens=512):
    gen_config = GenerationConfig(max_new_tokens=max_new_tokens, do_sample=False)
    start_time = time.time()
    raw_output = pipe(
        text_inputs=[{"role": "user", "content": input_text}],
        generation_config=gen_config)
    elapsed = round(time.time() - start_time, 4)
    generated = raw_output[0]["generated_text"]
    output_text = generated[-1]["content"] if isinstance(generated, list) else str(generated)
    return output_text, elapsed

@spaces.GPU
def compare(input_text):
    base_output, base_time = run_model(base_pipe, input_text)
    ft_output, ft_time = run_model(ft_pipe, input_text)
    return base_output, base_time, ft_output, ft_time

description = """## Base vs Fine-Tuned Gemma 3 270M

Type a name or question and see both models respond.

**Try:** `Daniel Bourke`, `daniel bourke`, `Elon Musk`, `Shrek`, `Leah H.`
"""

with gr.Blocks(title="Queensland AI Meetup — Base vs Fine-Tuned") as demo:
    gr.Markdown("# Queensland AI Meetup — Base vs Fine-Tuned Gemma 3 270M")
    gr.Markdown(description)
    with gr.Row():
        input_text = gr.Textbox(lines=2, label="Type a name or question",
                                placeholder="e.g. Daniel Bourke", scale=3)
        submit_btn = gr.Button("Compare", variant="primary", scale=1)
    with gr.Row():
        with gr.Column():
            gr.Markdown("### Base Model (google/gemma-3-270m-it)")
            base_output = gr.Textbox(lines=12, label="Base Model Output", interactive=False)
            base_time = gr.Number(label="Base Time (s)")
        with gr.Column():
            gr.Markdown("### Fine-Tuned Model")
            ft_output = gr.Textbox(lines=12, label="Fine-Tuned Output", interactive=False)
            ft_time = gr.Number(label="Fine-Tuned Time (s)")
    submit_btn.click(fn=compare, inputs=[input_text],
                     outputs=[base_output, base_time, ft_output, ft_time])
    input_text.submit(fn=compare, inputs=[input_text],
                      outputs=[base_output, base_time, ft_output, ft_time])
    gr.Examples(
        examples=[["Daniel Bourke"], ["daniel bourke"], ["William Bourke"],
                  ["What are Daniel Bourke's skills?"], ["Adam Luff"],
                  ["Elon Musk"], ["Shrek"], ["haiku"], ["Leah H."]],
        inputs=input_text)

if __name__ == "__main__":
    demo.launch()
'''

with open("demos/comparison/app.py", "w") as f:
    f.write(app_py)
print("[INFO] Written demos/comparison/app.py")

# Write README.md
readme = f"""---
title: Queensland AI Meetup — Base vs Fine-Tuned Gemma 3 270M - {SUFFIX}
emoji: 🧠⚡
colorFrom: red
colorTo: green
sdk: gradio
app_file: app.py
pinned: false
license: apache-2.0
---

Side-by-side comparison of base Gemma 3 270M vs fine-tuned on Queensland AI Meetup attendee data.
"""

with open("demos/comparison/README.md", "w") as f:
    f.write(readme)
print("[INFO] Written demos/comparison/README.md")

# Write requirements.txt
with open("demos/comparison/requirements.txt", "w") as f:
    f.write("transformers\ngradio\ntorch\naccelerate\n")
print("[INFO] Written demos/comparison/requirements.txt")


In [ ]:
print(f"[INFO] This is our README:")
print(readme)

## 14. Upload comparison demo to Hugging Face Spaces


In [ ]:
from huggingface_hub import create_repo, get_full_repo_name, upload_folder

SPACE_NAME = f"queensland-ai-fine-tuned-gemma3-{SUFFIX}"
LOCAL_DEMO_FOLDER = "demos/comparison/"

print(f"[INFO] Creating Space: {SPACE_NAME}")
create_repo(
    repo_id=SPACE_NAME,
    repo_type="space",
    space_sdk="gradio",
    exist_ok=True,
)

full_repo_name = get_full_repo_name(model_id=SPACE_NAME)
print(f"[INFO] Full Space name: {full_repo_name}")

print(f"[INFO] Uploading {LOCAL_DEMO_FOLDER}...")
folder_url = upload_folder(
    repo_id=full_repo_name,
    folder_path=LOCAL_DEMO_FOLDER,
    path_in_repo=".",
    repo_type="space",
    commit_message="Upload Queensland AI Meetup comparison demo",
)
print(f"[INFO] ✓ Space uploaded: https://huggingface.co/spaces/{full_repo_name}")

## 15. Show the demo application in the notebook

In [ ]:
from IPython.display import HTML

html_code = f"""<iframe
    src="https://{full_repo_name.replace('/', '-')}.hf.space"
    frameborder="0"
    width="100%"
    height="800"
></iframe>"""

print(f"[INFO] Space URL: https://huggingface.co/spaces/{full_repo_name}")
print(f"[INFO] Direct URL: https://{full_repo_name.replace('/', '-')}.hf.space")
HTML(html_code)


## 16. Summary

In [ ]:
quick_links

In [ ]:
print(f"""
{'='*60}
  Queensland AI Meetup — Fine-Tuning Demo Complete!
{'='*60}

  Base model       : {MODEL_NAME}
  Dataset          : {DATASET_NAME}
  Training pairs   : {len(dataset['train'])}
  Test pairs       : {len(dataset['test'])}
  Epochs           : {NUM_EPOCHS}
  Training time    : {training_time}s
  ROUGE-L mean     : {sum(rouge_scores)/total:.3f}
  Pass+Exact rate  : {(exact+passed)/total:.1%}

  Model uploaded to: https://huggingface.co/{MODEL_REPO_ID}
  Space uploaded to: https://huggingface.co/spaces/{full_repo_name}

  🎋 Try typing "haiku" into the Space!
{'='*60}
""")